In [ ]:
import os
print(os.listdir('/kaggle/input/'))


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

KAGGLE_INPUT_DIR = '/kaggle/input'
dataset_path = None

for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
 
    if 'Training' in dirs:
        dataset_path = os.path.join(root, 'Training')
        break
    elif any(d in dirs for d in ['glioma', 'meningioma', 'pituitary', 'notumor', 'no_tumor']):
        dataset_path = root
        break

print(f"Dataset Path Found: {dataset_path}")


train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 80% Train, 20% Validation
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    Flatten(),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dense(train_generator.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)


model.save('lumira_neuro_model.h5')
print("Done! Real 4-class 'lumira_neuro_model.h5' saved successfully!")

In [ ]:
from IPython.display import FileLink
FileLink('lumira_neuro_model.h5')